In [ ]:
!pip install --upgrade snowflake-connector-python


In [ ]:
import os
import yaml
import pandas as pd
import re # Added for RegEx stripping
from google import genai
from google.genai import types
import snowflake.connector
from google.colab import userdata

In [ ]:
def load_schema(file_path="IPL_SEMANTIC 4_8_2026, 1_30 AM.yaml"):
    """Loads the database schema from the YAML file."""
    try:
        with open(file_path, 'r') as f:
            return yaml.safe_load(f)
    except FileNotFoundError:
        print(f"[ERROR] Schema file '{file_path}' not found.")
        # Provide a minimal structure to avoid crashing if the file is missing
        return {'database_schema': {'tables': []}}

In [ ]:
DB_SCHEMA = load_schema()

In [ ]:
DB_SCHEMA

{'name': 'IPL_SEMANTIC',
 'tables': [{'name': 'BALL_BY_BALL',
   'description': 'The table contains ball-by-ball records of cricket match events across multiple seasons. Each record captures a single delivery, including the players involved (batter, bowler, and fielders), the teams batting and bowling, and the outcome of the delivery such as runs scored, extras, and dismissals.',
   'base_table': {'database': 'DEMO_DB',
    'schema': 'PUBLIC',
    'table': 'BALL_BY_BALL'},
   'dimensions': [{'name': 'BALL_NUMBER',
     'description': 'The sequential number of each ball bowled within an over in a cricket match.',
     'expr': 'BALL_NUMBER',
     'data_type': 'NUMBER',
     'sample_values': ['5', '3', '4']},
    {'name': 'BATSMAN_TYPE',
     'description': 'The batting hand orientation of the batsman.',
     'expr': 'BATSMAN_TYPE',
     'data_type': 'VARCHAR',
     'sample_values': ['Left hand Bat', 'Right hand Bat']},
    {'name': 'BATTER',
     'description': 'The name of the batter wh

In [ ]:
#Prompt

SCHEMA_PROMPT = f"""
You are an expert MySQL query generator.
Your task is to convert a natural language question into a single, valid, and executable MySQL query.

DATABASE SCHEMA:
---
{yaml.dump(DB_SCHEMA, indent=2)}
---

RULES:
1. Only output the SQL query. Do not include any explanations, comments, or surrounding text (like '```sql').
2. Ensure the query is syntactically correct for MySQL.
3. Use the exact table and column names provided in the schema.
"""

In [ ]:
def execute_sql_query(sql_query):
    """Connects to MySQL, executes the query, and returns the result."""
    # Ensure the query is not empty after stripping
    if not sql_query:
        print("\n[ERROR] Generated SQL query is empty after cleaning.")
        return None, None

    try:
        conn = snowflake.connector.connect(
            user='AFSDES',
            password=userdata.get('snowflake_password'), # Get password from Colab secrets
            account='OWNRHDE-ZU37157',  # e.g., 'xy12345.us-east-1'
            warehouse='COMPUTE_WH',
            database='NLP2SQL',
            schema='TEST_SCHEMA'
        )
        cur = conn.cursor()

        # Execute the query
        print(f"\n[INFO] Executing Clean SQL: {sql_query}")
        cur.execute(sql_query)

        # Get the results and column names
        rows = cur.fetchall()

        # Check if column names are available (e.g., SELECT statements)
        column_names = [i[0] for i in cur.description] if cur.description else None

        conn.close()

        return rows, column_names

    except Exception as e:
        print(f"\n[ERROR] An unexpected error occurred: {e}")
        return None, None

In [ ]:
client = genai.Client(api_key=userdata.get('secretdemo'))

In [ ]:
# --- Main Chatbot Logic ---
def chatbot_main(user_query):
    """Generates SQL using Gemini, executes it, and displays the result."""
    print(f"User Query: {user_query}")

    # 1. Generate SQL using Gemini
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[
                SCHEMA_PROMPT,
                f"Natural Language Question: {user_query}\n\nGenerated SQL:"
            ],
            config=types.GenerateContentConfig(
                temperature=0.4
            )
        )

        generated_sql = response.text.strip()


        # 2. **CRITICAL FIX:** Strip Markdown/Code wrappers from the SQL
        # This RegEx handles '```sql', '```', and any surrounding whitespace/newlines
        generated_sql = re.sub(r'```sql\s*|\s*```', '', generated_sql, flags=re.IGNORECASE).strip()
        print(f"[DEBUG] Raw Gemini Output: {generated_sql}")

        if not generated_sql:
            print("[ERROR] Failed to generate a valid, clean SQL query.")
            return

    except Exception as e:
        print(f"[ERROR] Gemini API call failed: {e}")
        return

    # 3. Execute the cleaned SQL
    rows, column_names = execute_sql_query(generated_sql)

    # 4. Display the result as a table
    if rows and column_names:
        print("\n✨ Chatbot Result Table ✨")
        # Use pandas for nice table formatting
        df = pd.DataFrame(rows, columns=column_names)
        print(df.to_markdown(index=False))
    elif rows is not None:
        # Handle cases like INSERT/UPDATE/DELETE or empty SELECT result
        print("\n✅ Query executed successfully. No data to display (e.g., INSERT, UPDATE, DELETE, or zero results).")
    else:
        print("\n❌ Could not retrieve or display data due to a prior error.")
    return df


In [ ]:
res = chatbot_main("which player had more sixes and which player had more wickets")

User Query: which player had more sixes and which player had more wickets
[DEBUG] Raw Gemini Output: (SELECT
    'Most Sixes' AS Statistic,
    BATTER AS Player,
    COUNT(*) AS Value
FROM
    BALL_BY_BALL
WHERE
    BATTER_RUNS = 6
GROUP BY
    BATTER
ORDER BY
    Value DESC
LIMIT 1)
UNION ALL
(SELECT
    'Most Wickets' AS Statistic,
    BOWLER AS Player,
    COUNT(*) AS Value
FROM
    BALL_BY_BALL
WHERE
    IS_WICKET = TRUE
    AND WICKET_KIND NOT IN ('run out', 'retired hurt', 'obstructing the field')
GROUP BY
    BOWLER
ORDER BY
    Value DESC
LIMIT 1);

[INFO] Executing Clean SQL: (SELECT
    'Most Sixes' AS Statistic,
    BATTER AS Player,
    COUNT(*) AS Value
FROM
    BALL_BY_BALL
WHERE
    BATTER_RUNS = 6
GROUP BY
    BATTER
ORDER BY
    Value DESC
LIMIT 1)
UNION ALL
(SELECT
    'Most Wickets' AS Statistic,
    BOWLER AS Player,
    COUNT(*) AS Value
FROM
    BALL_BY_BALL
WHERE
    IS_WICKET = TRUE
    AND WICKET_KIND NOT IN ('run out', 'retired hurt', 'obstructing the field')


In [ ]:
res

,STATISTIC,PLAYER,VALUE
0,Most Sixes,CH Gayle,359
1,Most Wickets,YS Chahal,221
